# CS781 Phase 1 — Notebook 4: Decoding lab & model analysis (no training)

Loads the weights of a finished NB3 run and answers three questions:

1. **Which decoding strategy maximises the official metric?** — a wider sweep than NB3 (n-gram block size,
   repetition penalty grid, low temperatures, **beam search**), all scored on the 400 local Supreme-Court prompts.
2. **Why is the model below the copy-the-prompt-tail baseline?** — per-example BERTScore of model vs copy-tail,
   precision/recall decomposition, and which kinds of prompts we lose on.
3. **How does validation loss relate to generation quality?** — scores the 25/50/75/100 % snapshots NB3 saved.

Finally it writes a `submission.zip` with the best decoding.

**Kaggle settings**: GPU T4 (one is enough), Internet ON, secret `HF`.
**Inputs**: NB1 output (eval sets, tokenizer) + the **NB3 run** to analyse (its output contains `runs/<run_name>/`).
Every result is cached in `decode_lab/<run_name>/`, so a re-run (with this notebook's previous output attached) skips finished work.

In [ ]:
!pip -q install tokenizers bert-score

In [ ]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

## 1. Configuration

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, csv, datetime, contextlib
SMOKE_TEST = os.environ.get("CS781_SMOKE") == "1"

RUN_NAME   = "A0_gpt2_110M_mix50_300M"     # which NB3 run to analyse
WEIGHTS    = "weights_final.pt"
N_EVAL     = 400                            # local Supreme-Court prompts used for the sweep
EVAL_SET   = "sc_random"
GEN_BATCH  = 48
DO_DIAGNOSIS, DO_SNAPSHOTS, MAKE_SUBMISSION = True, True, True
SNAPSHOT_N = 200                            # prompts per snapshot in the loss-vs-quality analysis

SWEEP = {
    # --- greedy + loop control
    "greedy_nr3":         dict(temperature=0.0, no_repeat_ngram=3),
    "greedy_nr4":         dict(temperature=0.0, no_repeat_ngram=4),
    "greedy_nr5":         dict(temperature=0.0, no_repeat_ngram=5),
    "greedy_rp1.05_nr4":  dict(temperature=0.0, rep_penalty=1.05, no_repeat_ngram=4),
    "greedy_rp1.15_nr4":  dict(temperature=0.0, rep_penalty=1.15, no_repeat_ngram=4),
    "greedy_rp1.3_nr4":   dict(temperature=0.0, rep_penalty=1.3, no_repeat_ngram=4),
    # --- low-temperature sampling
    "t0.3_nr4":           dict(temperature=0.3, no_repeat_ngram=4),
    "t0.5_nr4":           dict(temperature=0.5, no_repeat_ngram=4),
    "t0.7_p0.9_nr4":      dict(temperature=0.7, top_p=0.9, no_repeat_ngram=4),
    # --- beam search (most probable *sequence* rather than most probable next token)
    "beam3_nr4":          dict(beams=3, no_repeat_ngram=4),
    "beam5_nr4":          dict(beams=5, no_repeat_ngram=4),
    "beam3_rp1.15_nr4":   dict(beams=3, rep_penalty=1.15, no_repeat_ngram=4),
}
if os.environ.get("CS781_SMOKE") == "1":
    N_EVAL, SNAPSHOT_N, GEN_BATCH = 24, 12, 8
    SWEEP = {k: SWEEP[k] for k in ["greedy_nr4", "t0.5_nr4", "beam3_nr4"]}

OUT = ROOT / "decode_lab" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
restore_from_kaggle_inputs(["eval", f"decode_lab/{RUN_NAME}"])

def find_run_file(name):
    p = find_artifact(f"runs/{RUN_NAME}/{name}")
    assert p is not None, f"runs/{RUN_NAME}/{name} not found — attach the NB3 output of this run"
    return p

RUN_INFO = load_json(find_run_file("run_info.json"))
CFG = RUN_INFO["cfg"]
CFG["max_new_tokens"] = max(CFG.get("max_new_tokens", 256), 320)   # runs trained with older NB3 saved 256 (too short for citation-heavy prompts)
TOK_PATH = find_artifact(f"tokenizer/{CFG['tokenizer']}/tokenizer.json")
DECISIONS = load_json(find_artifact("data/decisions.json"))
assert TOK_PATH and DECISIONS, "attach NB1 output"
if not (ROOT / "eval" / "sc_random_prompts.json").exists():
    cands = load_json(find_artifact("eval/candidates_supreme_court.json"))
    save_json({f"sc_{c['doc_idx']}": {"text": c["rand_prompt"]} for c in cands}, ROOT / "eval" / "sc_random_prompts.json")
    save_json({f"sc_{c['doc_idx']}": {"text": c["rand_ref"]} for c in cands}, ROOT / "eval" / "sc_random_truth.json")
EVAL_PROMPTS = load_json(ROOT / "eval" / f"{EVAL_SET}_prompts.json")
EVAL_TRUTH   = load_json(ROOT / "eval" / f"{EVAL_SET}_truth.json")
IDS = list(EVAL_PROMPTS)[:N_EVAL]
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"run {RUN_NAME}: arch {CFG['arch']}, {RUN_INFO['n_params']/1e6:.1f}M params | eval {len(IDS)} prompts | device {DEV}")

## 2. Model, generation and metric code (copied verbatim from NB3 so behaviour is identical)

In [ ]:
# (identical to NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

with torch.device("meta"):          # count parameters without allocating or initialising anything (fork-safety)
    _m = build_model(CFG, 32768)
print(f"{CFG['arch']}: {count_params(_m)/1e6:.1f}M parameters (token embeddings {_m.wte.weight.numel()/1e6:.1f}M"
      + (f", learned positions {_m.wpe.weight.numel()/1e6:.2f}M)" if getattr(_m, "wpe", None) is not None else ")"))
if not SMOKE_TEST: assert 50e6 <= count_params(_m) <= 300e6, "outside the 50–300M rule"
del _m

In [ ]:
# (identical to NB3's generation cell)
def prepare_prompt_ids(tok, texts, ctx_len, max_new):
    texts = [normalize(t, DECISIONS["normalize"]) for t in texts]
    enc = tok.encode_batch(texts)
    keep = ctx_len - max_new
    return [e.ids[-keep:] for e in enc]

def reorder_cache(c, idx):
    """select / reorder batch rows of any cache structure (tensors, tuples, lists, dicts) — used by beam search"""
    if c is None: return None
    if torch.is_tensor(c): return c[idx]
    if isinstance(c, (list, tuple)): return type(c)(reorder_cache(x, idx) for x in c)
    if isinstance(c, dict): return {k: reorder_cache(v, idx) for k, v in c.items()}
    return c

class DecoderRunner:
    """Incremental generation for every decoder-only model: left-pad the prompts, run them once (prefill) to fill the
    cache, then feed one token at a time. prefill()/step() return the next-token logits (B, V) in fp32."""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):
            idx[r, L - len(ids_):] = torch.tensor(ids_); valid[r, L - len(ids_):] = True
        self.idx, self.valid = idx.to(device), valid.to(device)
    def prefill(self):
        L = self.idx.size(1); dev = self.device
        pos = (self.valid.long().cumsum(1) - 1).clamp(min=0)
        causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
        mask = (causal[None] & self.valid[:, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None]
        with self.amp:
            logits, _, self.cache = self.model(self.idx, pos=pos, mask=mask[:, None], use_cache=True, last_only=True)
        self.cur = pos[:, -1].clone()
        return logits[:, -1].float()
    def step(self, nxt):
        self.valid = torch.cat([self.valid, torch.ones((len(nxt), 1), dtype=torch.bool, device=self.device)], 1)
        self.cur = self.cur + 1
        with self.amp:
            logits, _, self.cache = self.model(nxt[:, None], pos=self.cur[:, None], mask=self.valid[:, None, None, :],
                                               cache=self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.cache = reorder_cache(self.cache, idx); self.valid = self.valid[idx]; self.cur = self.cur[idx]

def make_runner(model, rows, pad_id, device, amp):
    if hasattr(model, "make_runner"): return model.make_runner(rows, pad_id, device, amp)   # e.g. encoder-decoder
    return DecoderRunner(model, rows, pad_id, device, amp)

@torch.no_grad()
def generate(model, tok, prompts, dcfg, ctx_len, max_new, batch_size, device, target_words=CONT_WORDS, seed=0):
    model.eval()
    ids = prepare_prompt_ids(tok, prompts, ctx_len, max_new)
    order = np.argsort([len(x) for x in ids])
    outs = [None] * len(ids)
    n_special = 16
    pad_id = tok.token_to_id("<|pad|>")
    temp, top_k, top_p = dcfg.get("temperature", 0.0), dcfg.get("top_k", 0), dcfg.get("top_p", 1.0)
    rp, nr = dcfg.get("rep_penalty", 1.0), dcfg.get("no_repeat_ngram", 0)
    g = torch.Generator(device=device).manual_seed(seed)
    amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
    for b0 in range(0, len(ids), batch_size):
        bi = order[b0: b0 + batch_size]; B = len(bi)
        runner = make_runner(model, [ids[i] for i in bi], pad_id, device, amp)
        lg_next = runner.prefill()
        gen = [[] for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=device)
        seen = torch.zeros((B, lg_next.size(-1)), dtype=torch.bool, device=device)
        ngram_tables = [defaultdict(set) for _ in range(B)]
        for step in range(max_new):
            lg = lg_next
            lg[:, :n_special] = -float("inf")                      # never EOS / pad / reserved
            if rp != 1.0:
                lg = torch.where(seen, torch.where(lg > 0, lg / rp, lg * rp), lg)
            if nr > 1 and step >= nr - 1:
                for r in range(B):
                    banned = ngram_tables[r].get(tuple(gen[r][-(nr - 1):]))
                    if banned: lg[r, list(banned)] = -float("inf")
            if temp and temp > 0:
                lg = lg / temp
                if top_k: lg[lg < torch.topk(lg, top_k, dim=-1).values[:, -1:]] = -float("inf")
                if top_p < 1.0:
                    sp, si = torch.sort(lg, descending=True)
                    cp = torch.softmax(sp, -1).cumsum(-1)
                    rm = cp - torch.softmax(sp, -1) > top_p
                    sp[rm] = -float("inf"); lg = torch.full_like(lg, -float("inf")).scatter(1, si, sp)
                nxt = torch.multinomial(torch.softmax(lg, -1), 1, generator=g).squeeze(1)
            else:
                nxt = lg.argmax(-1)
            seen[torch.arange(B, device=device), nxt] = True
            nl = nxt.tolist()
            for r in range(B):
                if not done[r]:
                    gen[r].append(nl[r])
                    if nr > 1 and len(gen[r]) >= nr:
                        ngram_tables[r][tuple(gen[r][-nr:-1])].add(gen[r][-1])
            if step % 8 == 7 or step == max_new - 1:
                for r in range(B):
                    if not done[r] and len(tok.decode(gen[r]).split()) > target_words:
                        done[r] = True
                if bool(done.all()): break
            lg_next = runner.step(nxt)
        for r, i in enumerate(bi):
            words = tok.decode(gen[r]).split()[:target_words]
            outs[i] = " ".join(words) if words else "the"
    return outs

In [ ]:
# (identical to NB3's metric cell)
from collections import Counter as _C
def _lcs(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    masks = {}
    for j, t in enumerate(a): masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
    return bin(state).count("1")
def _rouge_l(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt: return 0.0
    l = _lcs(pt, rt)
    if not l: return 0.0
    pp, rr = l / len(pt), l / len(rt); return 2 * pp * rr / (pp + rr)
def _ngr(t, n): return _C(tuple(t[i:i + n]) for i in range(len(t) - n + 1)) if len(t) >= n else _C()
def _bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p); rl += len(r)
        for n in range(1, 5):
            pn, rn = _ngr(p, n), _ngr(r, n); totals[n - 1] += sum(pn.values())
            for gm, c in pn.items(): clipped[n - 1] += min(c, rn.get(gm, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0: return 0.0
    bp = 1.0 if pl > rl else math.exp(1.0 - rl / pl)
    return bp * math.exp(sum(math.log(x) for x in prec) / 4)

def score_phase1(truth, pred, with_bertscore=True):
    ids = list(truth)
    assert set(ids) == set(pred), "ID mismatch"
    out = {"n": len(ids),
           "rougeL": sum(_rouge_l(pred[i]["text"], truth[i]["text"]) for i in ids) / len(ids),
           "bleu4": _bleu4([pred[i]["text"].split() for i in ids], [truth[i]["text"].split() for i in ids])}
    if with_bertscore:
        try:
            import bert_score
            dev = "cuda" if torch.cuda.is_available() else "cpu"
            _, _, f = bert_score.score([pred[i]["text"].strip() for i in ids], [truth[i]["text"].strip() for i in ids],
                                       lang="en", batch_size=8 if dev == "cpu" else 64, verbose=False, device=dev)
            out["bertscore_f1"] = float(f.mean())
        except Exception as e:
            print("BERTScore unavailable:", e)
    return out

def repetition_rate(texts, n=4):
    """fraction of repeated 4-grams inside each output (diagnostic for degenerate loops)"""
    r = []
    for t in texts:
        w = t.split(); g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        r.append(1 - len(set(g)) / max(1, len(g)))
    return float(np.mean(r))

### Beam search
Keeps the `k` highest-scoring partial continuations (sum of token log-probabilities) instead of committing to the
single best token at each step. All beams have equal length (EOS is banned), so no length normalisation is needed.
The same loop controls as greedy apply per beam (`no_repeat_ngram`, `rep_penalty`). The KV-cache is re-ordered
each step so every surviving beam keeps the cache of the beam it came from. Uses the same per-architecture
"runner" as NB3's generate(), so it works for decoder-only, encoder-decoder, retention and hybrid models alike.

In [ ]:
@torch.no_grad()
def generate_beam(model, tok, prompts, dcfg, ctx_len, max_new, batch_size, device, target_words=CONT_WORDS):
    model.eval()
    K = dcfg["beams"]; rp = dcfg.get("rep_penalty", 1.0); nr = dcfg.get("no_repeat_ngram", 0)
    ids = prepare_prompt_ids(tok, prompts, ctx_len, max_new)
    order = np.argsort([len(x) for x in ids]); outs = [None] * len(ids)
    pad_id = tok.token_to_id("<|pad|>"); n_special = 16
    bsz = max(1, batch_size // K)
    amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
    for b0 in range(0, len(ids), bsz):
        bi = order[b0: b0 + bsz]; B = len(bi); R = B * K
        runner = make_runner(model, [ids[i] for i in bi for _ in range(K)], pad_id, device, amp)   # every prompt K times
        logits_last = runner.prefill()
        V = logits_last.size(-1)
        beam_scores = torch.zeros(B, K, device=device); beam_scores[:, 1:] = -1e9      # start from one live beam
        gens = [[] for _ in range(R)]; tables = [defaultdict(set) for _ in range(R)]
        seen = torch.zeros((R, V), dtype=torch.bool, device=device)
        finished = [None] * B
        for step in range(max_new):
            lp = torch.log_softmax(logits_last, -1)
            lp[:, :n_special] = -float("inf")
            if rp != 1.0:
                lp = torch.where(seen, lp * rp, lp)                 # log-probs are negative: multiplying by rp>1 penalises
            if nr > 1 and step >= nr - 1:
                for r in range(R):
                    banned = tables[r].get(tuple(gens[r][-(nr - 1):]))
                    if banned: lp[r, list(banned)] = -float("inf")
            cand = (beam_scores.view(R, 1) + lp).view(B, K * V)
            top_s, top_i = torch.topk(cand, K, dim=-1)
            src_beam = top_i // V; new_tok = top_i % V
            src_rows = (torch.arange(B, device=device)[:, None] * K + src_beam).view(-1)
            beam_scores = top_s
            sr, nt = src_rows.tolist(), new_tok.view(-1).tolist()
            gens = [gens[s] + [t] for s, t in zip(sr, nt)]
            new_tables = []
            for r, s in enumerate(sr):
                t = defaultdict(set, {k: set(v) for k, v in tables[s].items()}) if nr > 1 else tables[s]
                if nr > 1 and len(gens[r]) >= nr: t[tuple(gens[r][-nr:-1])].add(gens[r][-1])
                new_tables.append(t)
            tables = new_tables
            seen = seen[src_rows]; seen[torch.arange(R, device=device), new_tok.view(-1)] = True
            if step % 8 == 7 or step == max_new - 1:
                for b in range(B):
                    if finished[b] is None:
                        best = int(torch.argmax(beam_scores[b])); g = gens[b * K + best]
                        if len(tok.decode(g).split()) > target_words or step == max_new - 1:
                            finished[b] = g
                if all(f is not None for f in finished): break
            runner.reorder(src_rows)                               # every surviving beam keeps its parent's cache
            logits_last = runner.step(new_tok.view(-1))
        for b, i in enumerate(bi):
            g = finished[b] if finished[b] is not None else gens[b * K + int(torch.argmax(beam_scores[b]))]
            words = tok.decode(g).split()[:target_words]
            outs[i] = " ".join(words) if words else "the"
    return outs

def run_decoding(model, tok, prompts, dcfg):
    fn = generate_beam if dcfg.get("beams", 1) > 1 else generate
    return fn(model, tok, prompts, dcfg, CFG["ctx_len"], CFG["max_new_tokens"], GEN_BATCH, DEV)

### Official BERTScore with per-example results
`bert_score.BERTScorer(lang="en")` uses exactly the official settings (roberta-large, layer 17, no idf, no rescaling)
but loads the model once instead of on every call, and returns precision / recall / F1 for each example.

In [ ]:
_SCORER = None
def bertscore_items(preds, refs):
    global _SCORER
    try:
        import bert_score
    except ImportError:
        return None
    if _SCORER is None:
        _SCORER = bert_score.BERTScorer(lang="en", device="cuda" if torch.cuda.is_available() else "cpu",
                                        batch_size=64 if torch.cuda.is_available() else 8)
    P, R, F1 = _SCORER.score([p.strip() for p in preds], [r.strip() for r in refs])
    return {"P": P.numpy(), "R": R.numpy(), "F": F1.numpy()}

def full_score(ids, preds):
    refs = [EVAL_TRUTH[i]["text"] for i in ids]
    out = {"n": len(ids),
           "rougeL": float(np.mean([_rouge_l(p, r) for p, r in zip(preds, refs)])),
           "bleu4": _bleu4([p.split() for p in preds], [r.split() for r in refs]),
           "rep4": repetition_rate(preds)}
    bs = bertscore_items(preds, refs)
    if bs is not None:
        out.update({"bertscore_f1": float(bs["F"].mean()), "bertscore_p": float(bs["P"].mean()),
                    "bertscore_r": float(bs["R"].mean())})
    return out, bs

## 3. Decoding sweep

In [ ]:
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(TOK_PATH))
def load_weights(fname):
    m = build_model(CFG, TOK.get_vocab_size())
    sd = torch.load(find_run_file(fname), map_location="cpu", weights_only=False)
    m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    return m.to(DEV).eval()

MODEL = load_weights(WEIGHTS)
prompts = [EVAL_PROMPTS[i]["text"] for i in IDS]
res = load_json(OUT / "sweep.json", {})
for name, dcfg in SWEEP.items():
    if name in res: continue
    if time_left_h() < 1.0: print("time budget low — stopping sweep; re-run to continue"); break
    t0 = time.time()
    preds = run_decoding(MODEL, TOK, prompts, dcfg)
    sc, bs = full_score(IDS, preds)
    sc.update({"gen_seconds": time.time() - t0, "decoding": dcfg})
    res[name] = sc; save_json(res, OUT / "sweep.json")
    save_json(dict(zip(IDS, preds)), OUT / f"pred_{name}.json")
    if bs is not None: np.save(OUT / f"bs_{name}.npy", np.stack([bs["P"], bs["R"], bs["F"]]))
    print(f"{name:20s} BERTScore {sc.get('bertscore_f1', float('nan')):.4f}  ROUGE-L {sc['rougeL']:.4f}  "
          f"BLEU-4 {sc['bleu4']:.4f}  rep4 {sc['rep4']:.3f}  ({sc['gen_seconds']:.0f}s)")

# copy-tail baseline on exactly the same prompts (for the diagnosis below)
if "copy_tail" not in res:
    preds = [" ".join(normalize(EVAL_PROMPTS[i]["text"], DECISIONS["normalize"]).split()[-CONT_WORDS:]) for i in IDS]
    sc, bs = full_score(IDS, preds); sc["decoding"] = "baseline"; res["copy_tail"] = sc
    save_json(res, OUT / "sweep.json"); save_json(dict(zip(IDS, preds)), OUT / "pred_copy_tail.json")
    if bs is not None: np.save(OUT / "bs_copy_tail.npy", np.stack([bs["P"], bs["R"], bs["F"]]))

KEY = "bertscore_f1" if all("bertscore_f1" in r for r in res.values()) else "rougeL"
print(f"\n{'decoding':20s} {'BERTScore':>9s} {'(P':>7s} {'R)':>7s} {'ROUGE-L':>8s} {'BLEU-4':>7s} {'rep4':>6s} {'sec':>5s}")
for name, r in sorted(res.items(), key=lambda kv: -kv[1].get(KEY, 0)):
    print(f"{name:20s} {r.get('bertscore_f1', float('nan')):9.4f} {r.get('bertscore_p', float('nan')):7.4f} "
          f"{r.get('bertscore_r', float('nan')):7.4f} {r['rougeL']:8.4f} {r['bleu4']:7.4f} {r['rep4']:6.3f} "
          f"{r.get('gen_seconds', 0):5.0f}")
BEST = max([n for n in res if n != "copy_tail"], key=lambda n: res[n][KEY])
print(f"\nbest decoding by {KEY}: {BEST}")

## 4. Diagnosis: where does the model lose to copy-tail?
For every prompt we compare BERTScore F1 of the best decoding with copy-tail and relate the difference to simple
properties of the example:
* **ref∩prompt** – fraction of the reference's words that already appear in the prompt (how "copyable" the answer is)
* **digits** – fraction of reference words containing digits (citations, sections, dates)
* **prompt fertility** – tokens per word of the prompt (high = citation/number heavy)

BERTScore precision asks "is each generated token supported by the reference?", recall asks "is each reference
token covered by the output?" — a generic but fluent model typically loses on recall.

In [ ]:
if DO_DIAGNOSIS and (OUT / f"bs_{BEST}.npy").exists() and (OUT / "bs_copy_tail.npy").exists():
    m_bs, c_bs = np.load(OUT / f"bs_{BEST}.npy"), np.load(OUT / "bs_copy_tail.npy")
    pm, pc = load_json(OUT / f"pred_{BEST}.json"), load_json(OUT / "pred_copy_tail.json")
    diff = m_bs[2] - c_bs[2]
    def words(t): return [w.lower().strip(".,;:()[]\"'") for w in t.split()]
    feats = {"ref∩prompt": [], "digits": [], "prompt_fertility": [], "model∩prompt": []}
    for i in IDS:
        pw, rw, mw = set(words(EVAL_PROMPTS[i]["text"])), words(EVAL_TRUTH[i]["text"]), words(pm[i])
        feats["ref∩prompt"].append(np.mean([w in pw for w in rw]))
        feats["model∩prompt"].append(np.mean([w in pw for w in mw]))
        feats["digits"].append(np.mean([any(c.isdigit() for c in w) for w in rw]))
        txt = normalize(EVAL_PROMPTS[i]["text"], DECISIONS["normalize"])
        feats["prompt_fertility"].append(len(TOK.encode(txt).ids) / max(1, len(txt.split())))
    print(f"model ({BEST}) vs copy-tail on {len(IDS)} prompts")
    print(f"  mean F1  model {m_bs[2].mean():.4f} | copy-tail {c_bs[2].mean():.4f} | model wins on {np.mean(diff > 0):.1%} of prompts")
    print(f"  mean P   model {m_bs[0].mean():.4f} | copy-tail {c_bs[0].mean():.4f}")
    print(f"  mean R   model {m_bs[1].mean():.4f} | copy-tail {c_bs[1].mean():.4f}")
    print(f"  words of the output that also occur in the prompt: model {np.mean(feats['model∩prompt']):.3f} | "
          f"reference {np.mean(feats['ref∩prompt']):.3f} | copy-tail 1.000")
    print("\ncorrelation of (model − copy-tail) F1 with example properties:")
    for k in ["ref∩prompt", "digits", "prompt_fertility"]:
        print(f"  {k:18s} r = {np.corrcoef(feats[k], diff)[0, 1]:+.3f}   (mean {np.mean(feats[k]):.3f})")
    q = np.quantile(feats["ref∩prompt"], [0, 0.25, 0.5, 0.75, 1])
    print("\nF1 by how 'copyable' the reference is (quartiles of ref∩prompt):")
    for a, b in zip(q[:-1], q[1:]):
        sel = (np.array(feats["ref∩prompt"]) >= a) & (np.array(feats["ref∩prompt"]) <= b)
        print(f"  [{a:.2f}, {b:.2f}]  model {m_bs[2][sel].mean():.4f}  copy-tail {c_bs[2][sel].mean():.4f}  n={sel.sum()}")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].scatter(c_bs[2], m_bs[2], s=8, alpha=0.6); lo, hi = min(c_bs[2].min(), m_bs[2].min()), max(c_bs[2].max(), m_bs[2].max())
    ax[0].plot([lo, hi], [lo, hi], "k--", lw=1); ax[0].set_xlabel("copy-tail F1"); ax[0].set_ylabel(f"model F1 ({BEST})")
    ax[0].set_title("per-prompt BERTScore")
    ax[1].scatter(feats["ref∩prompt"], diff, s=8, alpha=0.6); ax[1].axhline(0, color="k", lw=1)
    ax[1].set_xlabel("fraction of reference words present in prompt"); ax[1].set_ylabel("model − copy-tail F1")
    ax[1].set_title("where the model loses")
    plt.tight_layout(); plt.savefig(OUT / "diagnosis.png", dpi=130); plt.show()
    worst = np.argsort(diff)[:2]; best_ = np.argsort(-diff)[:1]
    for tag, sel in [("LARGEST LOSS", worst), ("LARGEST WIN", best_)]:
        for j in sel:
            i = IDS[j]
            print(f"\n===== {tag}: {i}  model {m_bs[2][j]:.3f} vs copy-tail {c_bs[2][j]:.3f}\nPROMPT …{EVAL_PROMPTS[i]['text'][-250:]}"
                  f"\nMODEL  {pm[i][:500]}\nTRUE   {EVAL_TRUTH[i]['text'][:500]}")
    save_json({"diff_mean": float(diff.mean()), "win_rate": float(np.mean(diff > 0)),
               "corr": {k: float(np.corrcoef(feats[k], diff)[0, 1]) for k in ["ref∩prompt", "digits", "prompt_fertility"]},
               "model_overlap_with_prompt": float(np.mean(feats["model∩prompt"])),
               "ref_overlap_with_prompt": float(np.mean(feats["ref∩prompt"]))}, OUT / "diagnosis.json")
else:
    print("diagnosis skipped (needs BERTScore results)")

## 5. Validation loss vs generation quality across training (snapshots saved by NB3)
Same decoding for every snapshot (the best one above) on `SNAPSHOT_N` prompts. Val loss is read from NB3's
`metrics.jsonl` at the logged step closest to each snapshot.

In [ ]:
if DO_SNAPSHOTS:
    run_dir = find_run_file("run_info.json").parent
    snaps = sorted(run_dir.glob("weights_step*_*pct.pt"), key=lambda p: int(re.search(r"step(\d+)", p.name).group(1)))
    snaps = [(int(re.search(r"step(\d+)", p.name).group(1)), p.name) for p in snaps]
    snaps.append((RUN_INFO["total_steps"], WEIGHTS))
    recs = [json.loads(l) for l in open(run_dir / "metrics.jsonl")] if (run_dir / "metrics.jsonl").exists() else []
    vals = [r for r in recs if "val_supreme_court" in r]
    snap_res = load_json(OUT / "snapshots.json", {})
    sids = IDS[:SNAPSHOT_N]
    for step, fname in snaps:
        if fname in snap_res: continue
        if time_left_h() < 0.7: break
        m = MODEL if fname == WEIGHTS else load_weights(fname)
        preds = run_decoding(m, TOK, [EVAL_PROMPTS[i]["text"] for i in sids], SWEEP.get(BEST, {"temperature": 0.0, "no_repeat_ngram": 4}))
        sc, _ = full_score(sids, preds)
        near = min(vals, key=lambda r: abs(r["step"] - step)) if vals else {}
        snap_res[fname] = {"step": step, "tokens_M": step * RUN_INFO["tokens_per_step"] / 1e6,
                           "val_sc": near.get("val_supreme_court"), **sc}
        save_json(snap_res, OUT / "snapshots.json")
        if m is not MODEL: del m; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    rows = sorted(snap_res.values(), key=lambda r: r["step"])
    print(f"{'tokens':>8s} {'val SC loss':>11s} {'BERTScore':>10s} {'ROUGE-L':>8s} {'BLEU-4':>7s}")
    for r in rows:
        print(f"{r['tokens_M']:7.0f}M {r['val_sc'] if r['val_sc'] is not None else float('nan'):11.4f} "
              f"{r.get('bertscore_f1', float('nan')):10.4f} {r['rougeL']:8.4f} {r['bleu4']:7.4f}")
    if len(rows) > 1 and all(r["val_sc"] is not None for r in rows):
        import matplotlib.pyplot as plt
        k = "bertscore_f1" if "bertscore_f1" in rows[0] else "rougeL"
        plt.figure(figsize=(5.5, 4))
        plt.plot([r["val_sc"] for r in rows], [r[k] for r in rows], marker="o")
        for r in rows: plt.annotate(f"{r['tokens_M']:.0f}M", (r["val_sc"], r[k]), fontsize=8)
        plt.xlabel("Supreme-Court validation loss"); plt.ylabel(k); plt.gca().invert_xaxis()
        plt.title("lower loss → better generation?"); plt.tight_layout(); plt.savefig(OUT / "loss_vs_quality.png", dpi=130); plt.show()

## 6. Submission with the best decoding

In [ ]:
if MAKE_SUBMISSION:
    import zipfile
    files = load_json(find_artifact("data/files.json"))
    test = load_json(get_raw_file(files["test"], os.environ.get("CS781_LOCAL_RAW")))
    test_ids = list(test.keys()); test_texts = {i: extract_text(test[i]) for i in test_ids}
    sub_dir = OUT / f"submission_{BEST}"; sub_dir.mkdir(exist_ok=True)
    if not (sub_dir / "submission.zip").exists():
        t0 = time.time()
        preds = run_decoding(MODEL, TOK, [test_texts[i] for i in test_ids], SWEEP[BEST])
        pred = {i: {"text": p} for i, p in zip(test_ids, preds)}
        assert set(pred) == set(test_ids) and all(v["text"].strip() for v in pred.values())
        wc = [len(v["text"].split()) for v in pred.values()]
        assert max(wc) <= CONT_WORDS
        with open(sub_dir / "prediction.json", "w", encoding="utf-8") as f: json.dump(pred, f, ensure_ascii=False, indent=1)
        with zipfile.ZipFile(sub_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
            z.write(sub_dir / "prediction.json", arcname="prediction.json")
        print(f"{len(pred)} predictions, words min {min(wc)} mean {np.mean(wc):.1f} | {time.time()-t0:.0f}s")
    print("submission:", sub_dir / "submission.zip", "| local", KEY, f"{res[BEST][KEY]:.4f}")
    save_json({"run": RUN_NAME, "best_decoding": BEST, "best": res[BEST], "sweep": res}, OUT / "summary.json")